Credit to Martin Mulder from Soil Physics and Land Management Group (WUR) for providing the tools, methods and information needed in this initial soil parametrisation step. These include not only guidance towards the 'BRO Bodemkaart van Nederland', but also corresponding 'bodemprofielen' and 'staringreeks' tables. Lastly, the code seen in this notebook was transformed after his original code provided in R, which is deemed to derive soil profile curves on hydraulic conductivity and moisture content based on the 'staringreeks' table information.

In [8]:
import numpy as np
import pandas as pd
from pathlib import Path 
import sys
import os

In [9]:
# Defining Mualem van Genuchten functions (translated from MvG.R)

def get_wc_MvG(H, WCR, WCS, ALPHA, NPAR):
    """
    Calculate the water content (wc) based on pressure head (H) using the Mualem van Genuchten model.

    Parameters:
    H: Pressure head [cm].
    WCR: Residual water content [cm3 cm-3].
    WCS: Saturated water content [cm3 cm-3].
    ALPHA: Curve shape parameter [-].
    NPAR: Curve shape parameter [-].

    Returns:
    wc: Water content [cm3 cm-3].
    """
    m = 1 - 1 / NPAR
    wc = WCR + (WCS - WCR) / ((1 + (ALPHA * H)**NPAR)**m)
    return wc

def get_cond_MvG(H, ALPHA, NPAR, LAMBDA, KSAT):
    """
    Calculate the hydraulic conductivity (cond) as a function of pressure head (H) using the Mualem van Genuchten model.
    
    Parameters:
    H: Pressure head [cm].
    ALPHA: Curve shape parameter [-].
    NPAR: Curve shape parameter [-].
    LAMBDA: Exponent in hydraulic conductivity function [-].
    KSAT: Hydraulic conductivity of saturated soil[cm day-1].

    Returns:
    cond: Hydraulic conductivity [cm day-1].
    """
    m = 1 - 1 / NPAR
    ah = ALPHA * H 
    h1 = (1 + ah**NPAR)**m
    h2 = ah**(NPAR - 1)
    denom = (1 + ah**NPAR)**(m*(LAMBDA + 2))
    cond = KSAT * (h1 - h2)**2 / denom
    return cond   

def generate_pcse_tables(WCR, WCS, ALPHA, NPAR, LAMBDA, KSAT):
    """
    Generate SMTAB and CONTAB lookup tables for PCSE.

    PCSE expects:
        SMTAB: volumetric soil moisture content as a function of pF [log (cm); cm3 cm-3].
        CONTAB: 10-log hydraulic conductivity as a function of pF [log (cm); log (cm day-1)].
    
    Returns:
    smtab
    contab
    smw: soil moisture content at wilting point [cm3/cm3].
    smfcf: soil moisture content at field capacity [cm3/cm3].
    sm0: soil moisture content at saturation [cm3/cm3].
    k0: hydraulic conductivity at saturation [cm day-1]. 
    """
    
    # Set sequence of pressure head values
    pF_values = np.array([-1.0, 1.0, 1.3, 1.491, 1.7, 2.0, 2.4, 2.7, 3.0, 3.4, 3.7, 4.0, 4.204, 6.0]) #standard used in WOFOST files
    H_values = 10 ** pF_values # convert pF to pressure head in cm

    smtab = []
    contab = []

    for pF, H in zip(pF_values, H_values):
        theta = get_wc_MvG(H, WCR, WCS, ALPHA, NPAR)
        k = get_cond_MvG(H, ALPHA, NPAR, LAMBDA, KSAT)
        smtab.append((pF, round(theta, 3)))
        contab.append((pF, round(k, 3)))

    smw = round(get_wc_MvG(10**4.2, WCR, WCS, ALPHA, NPAR), 3) # soil moisture content at wilting point (pF=4.2)
    smfcf = round(get_wc_MvG(10**2.0, WCR, WCS, ALPHA, NPAR), 3) # soil moisture content at field capacity (pF=2.0)
    sm0 = round(WCS, 3) # soil moisture content at saturation
    k0 = round(KSAT, 3) # hydraulic conductivity at saturation
    
    return smtab, contab, smw, smfcf, sm0, k0
    

In [10]:
# Pipeline for gathering soil data and generating calibration site specific soil parameter file. 
folder = Path("/Users/paulianoprescu/Projects/wofost_miscanthus/miscanthus_calibration")

def pipeline():
    profielcode = 10110 # from BRO Bodemkaart van Nederland
    bodemprofiel_file = folder / "raw_data" / "soil" / "Bodemprofielen_20251009.csv"
    staringreeks_file = folder / "raw_data" / "soil" / "Staringreeks_2018_20251009.csv"
    output_dir = folder / "input_params" / "soil"

    # Step 1: Look up profile code in bodemprofiel file to find respective sfu code
    print(f"---Looking up profile code {profielcode} in Bodemprofielen_20251009.csv---")
    bp = pd.read_csv(bodemprofiel_file)
    rows = bp[bp["profielcode"] == profielcode].copy()

    if rows.empty:
        print(f"ERROR: No rows found with profile code {profielcode}")
        sys.exit(1)

    rows = rows.sort_values("ISOILLAY")    
    print(f"Found {len(rows)} soil layers for profile code {profielcode}:")
    print(rows[['profielcode', 'sfu', 'boven', 'onder', 'ISOILLAY', 'BDENS', 'PCLAY', 'PSILT', 'PSAND', 'ORGMAT', 'PH']].to_string(index=False))
    print()

    # Step 2: Look up sfu code in staringreeks file to find respective MvG parameters
    print(f"---Looking up sfu codes in Staringreeks_2018_20251009.csv---")
    sr = pd.read_csv(staringreeks_file)

    sfu_codes = rows["sfu"].unique()
    print(f"Unique sfu codes needed: {sfu_codes}")

    sr_params = {}
    for sfu in sfu_codes:
        row = sr[sr["sfu"] == sfu]
        if row.empty:
            print(f"ERROR: No rows found with sfu code {sfu}")
            sys.exit(1)
            continue 
        row = row.iloc[0]
        sr_params[sfu] = {
            "ORES": row["ORES"],
            "OSAT": row["OSAT"],
            "ALFA": row["ALFA"],
            "NPAR": row["NPAR"],
            "KSATFIT": row["KSATFIT"],
            "LEXP": row["LEXP"]}
        
        print(f"\n  {sfu}:")
        for k, v in sr_params[sfu].items():
            print(f"    {k:10s} = {v:.6f}")
    print()
    
    # Step 3: Generate WOFOST tables per soil layer
    # For a layered profile, for SMTAB/CONTAB I choose to use the topsoil layer's parameters, since these represent the water balance in the rooting zone.
    # Here tables are generated for each layer, but only the top layer's tables are saved in the output soil parameter file.

    print("---Generating PCSE Soil Parameters for all Soil Layers---")

    all_layer_results = []
    for i, row in rows.iterrows():
        sfu = row["sfu"]
        p = sr_params[sfu]

        smtab, contab, smw, smfcf, sm0, k0 = generate_pcse_tables(
            WCR = p["ORES"],
            WCS = p["OSAT"],
            ALPHA = p["ALFA"],
            NPAR = p["NPAR"],
            LAMBDA = p["LEXP"],
            KSAT = p["KSATFIT"]
        )

        result = { 
            'layer': row['ISOILLAY'],
            'depth': f"{row['boven']}-{row['onder']} cm",
            'sfu': sfu,
            'smtab': smtab,
            'contab': contab,
            'smw': smw,
            'smfcf': smfcf,
            'sm0': sm0,
            'k0': k0
        }
        all_layer_results.append(result)

        print(f"Layer {row['ISOILLAY']} ({row['boven']}-{row['onder']} cm, sfu={sfu}):")
        print(f"  SMW (wilting point, pF=4.2)  = {smw:.3f} cm3/cm3")
        print(f"  SMFCF (field capacity, pF=2) = {smfcf:.3f} cm3/cm3")
        print(f"  SM0 (saturation)             = {sm0:.3f} cm3/cm3")
        print(f"  K0 (Ksat)                    = {k0:.3f} cm/day")
        print()

    # Step 4: Write & Save Soil Parameter File
    # Use topsoil layer for the main SMTAB/CONTAB
    topsoil = all_layer_results[0]

    # Maximum percolation rate rootzone (SOPE) taken from topsoil - ASSUMPTION!
    sope = round(topsoil['k0'], 2)

    # Maximum percolation rate subsoil (KSUB) taken from subsoil - !
    subsoil = all_layer_results[-1]
    ksub = round(subsoil['k0'], 2)

    # Critical soil air content for aeration [cm3/cm3] assumed based on example coarse soil file; site specific topsoil layer has almost 90& sand, thus likely more aeration
    crairc = 0.090 # Similar assumptions also made soil workability parameters

    # Soil depth from profile
    max_depth = rows['onder'].max()

    # Format SMTAB
    smtab_lines = []
    for i, (pF, theta) in enumerate(topsoil['smtab']):
        if i == 0:
            smtab_lines.append(f"SMTAB    = {pF:7.3f}, {theta:7.3f},     ! vol. soil moisture content")
        elif i == 1:
            smtab_lines.append(f"           {pF:7.3f}, {theta:7.3f},     ! as function of pF [log (cm); cm3 cm-3]")
        else:
            smtab_lines.append(f"           {pF:7.3f}, {theta:7.3f},")
    smtab_lines[-1] = smtab_lines[-1].rstrip(',')

    # Format CONTAB
    contab_lines = []
    for i, (pF, k) in enumerate(topsoil['contab']):
        if i == 0:
            contab_lines.append(f"CONTAB    = {pF:7.3f}, {k:7.3f},       ! 10-log hydraulic conductivity")
        elif i == 1:
            contab_lines.append(f"            {pF:7.3f}, {k:7.3f},       ! as function of pF [log (cm); log (cm day-1)]")
        else:
            contab_lines.append(f"            {pF:7.3f}, {k:7.3f},")
    contab_lines[-1] = contab_lines[-1].rstrip(',')

    # Build Soil File 
    topsoil_sfu = topsoil['sfu']
    soil_file = f"""** Generated from Bodemkaart van Nederland
** Profielcode: {profielcode}
** Topsoil sfu: {topsoil_sfu} (Staringreeks 2018)
** Generated by: generate_wofost_soil.py
**
** SOIL PARAMTER FILE for use in PCSE
**
 
SOLNAM='{profielcode}-{topsoil_sfu}'
 
** physical soil characteristics
 
** soil water retention
{chr(10).join(smtab_lines)}
 
SMW      = {topsoil['smw']:7.3f}  ! soil moisture content at wilting point [cm3/cm3]
SMFCF    = {topsoil['smfcf']:7.3f}  ! soil moisture content at field capacity [cm3/cm3]
SM0      = {topsoil['sm0']:7.3f}  ! soil moisture content at saturation [cm3/cm3]
CRAIRC   = {crairc:7.3f}  ! critical soil air content for aeration [cm3/cm3]
 
** hydraulic conductivity
{chr(10).join(contab_lines)}
 
K0       = {topsoil['k0']:10.3f} ! hydraulic conductivity of saturated soil [cm day-1]
SOPE     = {sope:7.2f}   ! maximum percolation rate root zone [cm day-1]
KSUB     = {ksub:7.2f}   ! maximum percolation rate subsoil [cm day-1]
 
** soil workability parameters
SPADS    =   0.800  ! 1st topsoil seepage parameter deep seedbed 
SPODS    =   0.040  ! 2nd topsoil seepage parameter deep seedbed
SPASS    =   0.900  ! 1st topsoil seepage parameter shallow seedbed
SPOSS    =   0.070  ! 2nd topsoil seepage parameter shallow seedbed
DEFLIM   =   0.000  ! required moisture deficit deep seedbed
 
** soil depth
RDMSOL   = {max_depth}
"""
    
    # Write output
    os.makedirs(output_dir, exist_ok=True)
    outpath = os.path.join(output_dir, f"soil_{profielcode}.soil")
    with open(outpath, 'w') as f:
        f.write(soil_file)
    print(f"Soil parameter file written to: {outpath}")

    return outpath

pipeline()

---Looking up profile code 10110 in Bodemprofielen_20251009.csv---
Found 3 soil layers for profile code 10110:
 profielcode sfu  boven  onder  ISOILLAY       BDENS  PCLAY  PSILT  PSAND  ORGMAT  PH
       10110 B02      0     25         1 1373.441228   0.04   0.08   0.88   0.055 4.8
       10110 O02     25     50         2 1670.625666   0.03   0.09   0.88   0.002 4.8
       10110 O01     50    120         3 1678.648474   0.02   0.06   0.92   0.002 5.3

---Looking up sfu codes in Staringreeks_2018_20251009.csv---
Unique sfu codes needed: <StringArray>
['B02', 'O02', 'O01']
Length: 3, dtype: str

  B02:
    ORES       = 0.020000
    OSAT       = 0.433878
    ALFA       = 0.021645
    NPAR       = 1.348770
    KSATFIT    = 83.241635
    LEXP       = 7.202077

  O02:
    ORES       = 0.020000
    OSAT       = 0.387064
    ALFA       = 0.016083
    NPAR       = 1.524418
    KSATFIT    = 22.761756
    LEXP       = 2.439662

  O01:
    ORES       = 0.010000
    OSAT       = 0.365847
    ALFA  

'/Users/paulianoprescu/Projects/wofost_miscanthus/miscanthus_calibration/input_params/soil/soil_10110.soil'